# Colab Evaluation: v4.1 vs v3.1 Adaptive Operator Benchmark

This notebook benchmarks the fine-tuned **Qwen3.5-9B adaptive operator** model (v4.1)
against the v3.1 baseline using the same evaluation suite that produced the H100 benchmarks.

## Two Inference Modes

| Mode | Description | Fairness |
|------|-------------|----------|
| `together_api` | Together AI serverless API (fp16 on H100) | **Fair comparison** with v3.1 H100 benchmarks |
| `local_t4` | Local T4 GPU inference (4-bit NF4 quantization) | Results may differ slightly due to quantization |

> **Note**: The user specified "dont benchmark in 4 bit" because the v3.1 benchmarks were
> done on H100 in fp16. Since T4 only has 16GB VRAM (a 9B model in fp16 needs ~18GB),
> the `local_t4` mode loads the model in 4-bit NF4. For a **fair comparison**, use
> `together_api` mode which runs both models in fp16 on H100 via the Together AI API.

## Evaluation Dimensions

1. **Control token accuracy** — does the model emit the right token ([FAST], [THINK], etc.)?
2. **Tool selection accuracy** — does the model choose the right tool?
3. **Tool call format validity** — are tool calls valid JSON with correct parameters?
4. **Task completion** — can the model actually complete tasks end-to-end?

## Statistical Analysis

- Wilson 95% confidence intervals for pass rates
- Fisher's exact test for significance
- Cohen's d effect size
- Bonferroni correction for multiple category comparisons
- Per-category breakdown with significance flags

## Cell 1: Setup

Install required packages.

In [ ]:
!pip install -q transformers peft bitsandbytes datasets accelerate torch openai

## Cell 2: Mount Google Drive

Mount Drive to access the model files and utils.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3: Configuration

Set the evaluation mode and model paths. Choose `together_api` for a fair fp16 comparison
with the v3.1 H100 benchmarks, or `local_t4` for local 4-bit inference on the T4 GPU.

In [ ]:
# === Eval Configuration ===
EVAL_MODE = "together_api"  # "together_api" (H100 fp16, fair comparison) or "local_t4" (4-bit)

# For together_api mode:
TOGETHER_API_KEY = ""  # User fills in their Together AI API key
TOGETHER_BASE_URL = "https://api-inference.together.ai/v1"
V41_MODEL = ""  # e.g. "davidnicholsop-199a/adaptive-operator-v4-1"
V31_MODEL = "davidnicholsop-199a/adaptive-operator-v3-1-eval"

# For local_t4 mode:
MODEL_PATH = "/content/drive/MyDrive/adaptive_operator_v4/models/merged"

# Test categories to run (None = all categories)
TEST_CATEGORIES = None  # or ["simple_file_ops", "complex_coding", ...]

# Inference settings
MAX_TOKENS = 2048
TEMPERATURE = 0.0  # Greedy decoding for reproducibility
NUM_SEEDS = 3  # Run each test case 3 times for variance estimation

# Output
PROJECT_DIR = '/content/drive/MyDrive/adaptive_operator_v4'
RESULTS_PATH = f"{PROJECT_DIR}/eval_results.json"

# Import the utils module from Drive
import sys
sys.path.insert(0, PROJECT_DIR)

print(f"Evaluation mode: {EVAL_MODE}")
if EVAL_MODE == "together_api":
    if not TOGETHER_API_KEY:
        print("WARNING: TOGETHER_API_KEY is empty. Set it in the cell above.")
    if not V41_MODEL:
        print("WARNING: V41_MODEL is empty. Set it to your v4.1 model name.")
    print(f"  v4.1 model: {V41_MODEL}")
    print(f"  v3.1 model: {V31_MODEL}")
    print(f"  Inference: fp16 on H100 (fair comparison with v3.1 benchmarks)")
elif EVAL_MODE == "local_t4":
    print(f"  Model path: {MODEL_PATH}")
    print(f"  Inference: 4-bit NF4 on T4 (results may differ slightly from fp16)")
    print(f"  NOTE: For fair comparison with v3.1 H100 benchmarks, use together_api mode.")
print(f"  Max tokens: {MAX_TOKENS}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Seeds per case: {NUM_SEEDS}")

## Cell 4: Load Test Cases

Import the test cases. We use the `colab_eval_utils` module which contains the
tool schemas, system prompt builder, and response parsing functions. The test
cases are defined inline (ported from the eval suite) so the notebook is
self-contained on Colab.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Any

from colab_eval_utils import (
    CONTROL_TOKENS,
    parse_control_token,
    parse_tool_calls,
    build_system_prompt,
    get_tool_schemas,
    wilson_interval,
    fisher_exact_test,
    cohens_d,
    statistical_report,
    format_summary_table,
    format_report_text,
)

# --------------------------------------------------------------------------- #
# Test case definitions (ported from eval/test_cases.py)                       #
# --------------------------------------------------------------------------- #

# Helper eval functions

def _file_contains(workspace, rel_path, needle):
    p = workspace / rel_path
    if not p.is_file():
        return False
    return needle in p.read_text(encoding="utf-8", errors="ignore")

def _file_exists(workspace, rel_path):
    return (workspace / rel_path).is_file()

def _file_matches(workspace, rel_path, pattern):
    p = workspace / rel_path
    if not p.is_file():
        return False
    return re.search(pattern, p.read_text(encoding="utf-8", errors="ignore")) is not None

def _tool_was_used(metadata, tool_name):
    return tool_name in metadata.get("tools_used", [])

def _answer_contains(answer, needle):
    return needle.lower() in answer.lower()

# Setup functions

def _setup_read_file(ws):
    (ws / "target.txt").write_text("The secret value is 42.\nRemember this number.", encoding="utf-8")

def _setup_list_files(ws):
    (ws / "alpha.txt").write_text("alpha content", encoding="utf-8")
    (ws / "beta.txt").write_text("beta content", encoding="utf-8")
    (ws / "gamma.txt").write_text("gamma content", encoding="utf-8")

def _setup_find_python(ws):
    (ws / "main.py").write_text("print('hello')\n", encoding="utf-8")
    (ws / "util.py").write_text("def helper():\n    pass\n", encoding="utf-8")
    (ws / "readme.md").write_text("# Project", encoding="utf-8")

def _setup_edit_file(ws):
    (ws / "todo.py").write_text("# TODO: implement\n\ndef foo():\n    pass\n", encoding="utf-8")

def _setup_copy_file(ws):
    (ws / "original.txt").write_text("This is the original content.\nLine 2.\nLine 3.", encoding="utf-8")

def _setup_count_lines(ws):
    (ws / "data.txt").write_text("\n".join(f"line {i}" for i in range(10)) + "\n", encoding="utf-8")

def _setup_append_file(ws):
    (ws / "log.txt").write_text("initial entry\n", encoding="utf-8")

def _setup_syntax_error(ws):
    (ws / "broken.py").write_text("def foo()\n    return 42\n", encoding="utf-8")

def _setup_test_failure(ws):
    (ws / "calc.py").write_text("def multiply(a, b):\n    return a + b\n", encoding="utf-8")
    (ws / "test_calc.py").write_text(
        "from calc import multiply\n\ndef test_multiply():\n    assert multiply(3, 4) == 12\n", encoding="utf-8"
    )

def _setup_import_error(ws):
    (ws / "imports.py").write_text(
        "import nonexistent_module\n\nresult = nonexistent_module.do_something()\n", encoding="utf-8"
    )

def _setup_git_conflict(ws):
    (ws / "README.md").write_text(
        "# Project\n\n<<<<<<< HEAD\nThis is the main branch version.\n=======\nThis is the feature branch version.\n>>>>>>> feature-branch\n",
        encoding="utf-8",
    )

def _setup_type_error(ws):
    (ws / "types.py").write_text(
        "def combine(x, y):\n    return x + y\n\nresult = combine('hello', 42)\n", encoding="utf-8"
    )

def _setup_verify_tests(ws):
    (ws / "math_utils.py").write_text(
        "def add(a, b):\n    return a + b\n\ndef subtract(a, b):\n    return a - b\n", encoding="utf-8"
    )
    (ws / "test_math.py").write_text(
        "from math_utils import add, subtract\n\ndef test_add():\n    assert add(2, 3) == 5\n\ndef test_subtract():\n    assert subtract(5, 3) == 2\n",
        encoding="utf-8",
    )

def _setup_verify_file(ws):
    (ws / "config.yaml").write_text("name: my_app\nversion: 1.0.0\ndatabase:\n  host: localhost\n  port: 5432\n", encoding="utf-8")

def _setup_verify_syntax(ws):
    (ws / "check_me.py").write_text("def hello():\n    print('hello world')\n\nhello()\n", encoding="utf-8")

def _setup_verify_checksum(ws):
    (ws / "important.bin").write_text("critical data that needs verification\n" * 10, encoding="utf-8")

def _setup_verify_deps(ws):
    (ws / "app.py").write_text(
        "import os\nimport sys\nimport json\nimport nonexistent_pkg\n\nprint('hello')\n", encoding="utf-8"
    )

def _setup_verify_formatting(ws):
    (ws / "style.py").write_text(
        "def foo(  x,y ):\n    if x>0:\n        return  y\n    else:\n        return 0\n", encoding="utf-8"
    )

def _setup_verify_secrets(ws):
    (ws / "settings.py").write_text(
        'API_KEY = "sk-1234567890abcdef"\nSECRET = "my-secret-token"\nDB_PASSWORD = "password123"\n', encoding="utf-8"
    )

def _setup_refactor(ws):
    (ws / "monolith.py").write_text(
        "class User:\n    def __init__(self, name):\n        self.name = name\n\n"
        "class Product:\n    def __init__(self, title):\n        self.title = title\n\n"
        "class Order:\n    def __init__(self, user, product):\n        self.user = user\n        self.product = product\n",
        encoding="utf-8",
    )

def _setup_empty_file(ws):
    (ws / "empty.txt").write_text("", encoding="utf-8")

def _setup_unicode_file(ws):
    (ws / "unicode.txt").write_text("H\u00e9llo W\u00f6rld\n\u65e5\u672c\u8a9e\u30c6\u30b9\u30c8\nEmoji: \U0001f389\n", encoding="utf-8")

def _setup_large_file(ws):
    (ws / "large.txt").write_text("\n".join(f"line {i}: data content here" for i in range(1000)) + "\n", encoding="utf-8")

def _setup_special_chars(ws):
    (ws / "file with spaces & symbols!.txt").write_text("Content with special filename\n", encoding="utf-8")

def _setup_nested_dirs(ws):
    target = ws / "a" / "b" / "c" / "d"
    target.mkdir(parents=True, exist_ok=True)
    (target / "target.txt").write_text("You found me!\n", encoding="utf-8")

def _setup_binary_file(ws):
    (ws / "data.bin").write_bytes(bytes(range(256)) * 4)

def _setup_wrong_path(ws):
    (ws / "actual_location").mkdir(exist_ok=True)
    (ws / "actual_location" / "config.json").write_text('{"found": true}\n', encoding="utf-8")

# Eval functions

def _eval_create_file(ws, answer, meta):
    if _file_contains(ws, "hello.txt", "Hello, World!"):
        return True, "File created with correct content"
    if _file_exists(ws, "hello.txt"):
        return False, "File exists but content is wrong"
    return False, "hello.txt was not created"

def _eval_read_and_report(ws, answer, meta):
    if _answer_contains(answer, "42"):
        return True, "Model read the file and reported the secret value"
    if not _tool_was_used(meta, "file_read"):
        return False, "Model did not use file_read tool"
    return False, "Model read the file but did not report the secret (42)"

def _eval_list_files(ws, answer, meta):
    if _tool_was_used(meta, "file_list") or _tool_was_used(meta, "shell"):
        if _answer_contains(answer, "alpha") or _answer_contains(answer, "beta"):
            return True, "Model listed files and reported them"
        return False, "Model listed files but did not report contents"
    return False, "Model did not use file_list or shell to list files"

def _eval_find_python_files(ws, answer, meta):
    if _tool_was_used(meta, "find_file") or _tool_was_used(meta, "grep") or _tool_was_used(meta, "shell"):
        if _answer_contains(answer, "main.py") or _answer_contains(answer, "util.py"):
            return True, "Model found the Python files"
        return False, "Model searched but did not report the .py files"
    return False, "Model did not use a search tool"

def _eval_write_config(ws, answer, meta):
    if _file_matches(ws, "config.json", r'"name"\s*:\s*"test_app"'):
        return True, "config.json created with correct name field"
    if _file_exists(ws, "config.json"):
        return False, "config.json exists but does not have name=test_app"
    return False, "config.json was not created"

def _eval_edit_file(ws, answer, meta):
    p = ws / "todo.py"
    if not p.is_file():
        return False, "todo.py not found (setup may have failed)"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "TODO: implement" in content:
        return False, "TODO comment still present"
    if "implemented" in content.lower() or "done" in content.lower():
        return True, "TODO was replaced"
    return False, f"File was edited but replacement content is unexpected: {content[:200]}"

def _eval_create_directory(ws, answer, meta):
    if (ws / "src" / "components").is_dir():
        return True, "Directory structure src/components created"
    return False, "src/components directory was not created"

def _eval_copy_file(ws, answer, meta):
    if _file_exists(ws, "backup.txt") and _file_exists(ws, "original.txt"):
        orig = (ws / "original.txt").read_text(encoding="utf-8", errors="ignore")
        backup = (ws / "backup.txt").read_text(encoding="utf-8", errors="ignore")
        if orig == backup:
            return True, "File copied successfully with matching content"
        return False, "backup.txt exists but content does not match original"
    return False, "Copy did not complete"

def _eval_count_lines(ws, answer, meta):
    if _answer_contains(answer, "10"):
        return True, "Model correctly reported 10 lines"
    return False, f"Model did not report 10 lines. Answer: {answer[:200]}"

def _eval_append_to_file(ws, answer, meta):
    p = ws / "log.txt"
    if not p.is_file():
        return False, "log.txt not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "initial entry" in content and "new entry" in content.lower():
        return True, "New entry was appended to the file"
    if "new entry" in content.lower():
        return True, "New entry was added"
    return False, "New entry was not appended"

def _eval_binary_search(ws, answer, meta):
    p = ws / "binary_search.py"
    if not p.is_file():
        return False, "binary_search.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_func = "def binary_search" in content or "def binarySearch" in content
    has_mid = "mid" in content.lower()
    has_loop = "while" in content or "recursion" in content or "return binary_search" in content
    if has_func and has_mid and has_loop:
        return True, "Binary search implementation looks correct"
    return False, f"Implementation missing key elements. Content: {content[:300]}"

def _eval_flatten_function(ws, answer, meta):
    p = ws / "flatten.py"
    if not p.is_file():
        return False, "flatten.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_func = "def flatten" in content
    has_iter = "isinstance" in content or "Iterable" in content or "for" in content
    if has_func and has_iter:
        return True, "Flatten function implementation looks correct"
    return False, f"Implementation missing key elements. Content: {content[:300]}"

def _eval_rest_api(ws, answer, meta):
    p = ws / "app.py"
    if not p.is_file():
        for name in ("main.py", "server.py", "api.py"):
            if (ws / name).is_file():
                p = ws / name
                break
    if not p.is_file():
        return False, "No API server file was created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_route = "@app.route" in content or "@app.get" in content or "@app.post" in content or "add_route" in content
    has_users = "users" in content.lower() or "/users" in content
    if has_route and has_users:
        return True, "REST API with /users endpoint created"
    return False, f"API file missing route or /users endpoint. Content: {content[:300]}"

def _eval_class_hierarchy(ws, answer, meta):
    p = ws / "shapes.py"
    if not p.is_file():
        return False, "shapes.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_base = "class Shape" in content or "class BaseShape" in content
    has_circle = "class Circle" in content
    has_rect = "class Rectangle" in content or "class Rect" in content
    has_area = "def area" in content
    if has_base and has_circle and has_rect and has_area:
        return True, "Class hierarchy with Shape, Circle, Rectangle, and area() created"
    missing = []
    if not has_base: missing.append("base Shape class")
    if not has_circle: missing.append("Circle class")
    if not has_rect: missing.append("Rectangle class")
    if not has_area: missing.append("area() method")
    return False, f"Missing: {', '.join(missing)}"

def _eval_decorator_pattern(ws, answer, meta):
    p = ws / "decorators.py"
    if not p.is_file():
        return False, "decorators.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_decorator = "def retry" in content or "def memoize" in content or "def cache" in content
    has_wrapper = "wrapper" in content or "inner" in content
    if has_decorator and has_wrapper:
        return True, "Decorator pattern implemented"
    return False, f"Decorator implementation incomplete. Content: {content[:300]}"

def _eval_data_pipeline(ws, answer, meta):
    p = ws / "pipeline.py"
    if not p.is_file():
        return False, "pipeline.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_read = "open(" in content or "read_csv" in content or "json.load" in content
    has_transform = "def transform" in content or "def process" in content or "def clean" in content
    has_write = "write" in content or "dump" in content or "to_csv" in content
    if has_read and has_transform and has_write:
        return True, "Data pipeline with read/transform/write stages created"
    return False, f"Pipeline missing stages. Content: {content[:300]}"

def _eval_test_suite(ws, answer, meta):
    test_files = list(ws.glob("test_*.py")) + list(ws.glob("*_test.py"))
    if not test_files:
        return False, "No test files were created"
    content = test_files[0].read_text(encoding="utf-8", errors="ignore")
    has_assert = "assert" in content
    has_test_func = "def test_" in content
    if has_assert and has_test_func:
        return True, f"Test suite created: {test_files[0].name}"
    return False, f"Test file missing assertions or test functions. Content: {content[:300]}"

def _eval_algorithm_optimize(ws, answer, meta):
    p = ws / "optimized.py"
    if not p.is_file():
        return False, "optimized.py was not created"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_hash = "dict" in content or "{}" in content or "set(" in content
    has_efficient = "O(n)" in content or "hash" in content.lower() or "set" in content
    if has_hash or has_efficient:
        return True, "Optimized solution uses efficient data structure"
    return False, f"Solution may not be optimized. Content: {content[:300]}"

def _eval_recover_missing_file(ws, answer, meta):
    if _file_exists(ws, "data.txt"):
        return True, "Model recovered from missing file error by creating it"
    if _tool_was_used(meta, "file_read") and _tool_was_used(meta, "file_write"):
        return True, "Model attempted read then write (recovery behavior)"
    return False, "Model did not recover"

def _eval_recover_syntax_error(ws, answer, meta):
    p = ws / "broken.py"
    if not p.is_file():
        return False, "broken.py not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "def foo()" in content and ":" not in content.split("def foo()")[1].split("\n")[0]:
        return False, "Syntax error not fixed"
    try:
        compile(content, "broken.py", "exec")
        return True, "Syntax error was fixed"
    except SyntaxError as e:
        return False, f"File still has syntax error: {e}"

def _eval_recover_wrong_path(ws, answer, meta):
    if _answer_contains(answer, "found") or _answer_contains(answer, "content"):
        if _tool_was_used(meta, "file_read") or _tool_was_used(meta, "grep"):
            return True, "Model recovered from wrong path and found the file"
    return False, "Model did not recover from wrong path"

def _eval_recover_test_failure(ws, answer, meta):
    p = ws / "calc.py"
    if not p.is_file():
        return False, "calc.py not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "*" in content or "multiply" in content.lower():
        return True, "Model fixed the function to multiply"
    return False, "Function was not fixed to multiply"

def _eval_recover_permission_denied(ws, answer, meta):
    if _answer_contains(answer, "permission") or _answer_contains(answer, "denied") or _answer_contains(answer, "alternative"):
        return True, "Model acknowledged the permission error and tried alternative"
    if _tool_was_used(meta, "shell") and len(meta.get("tools_used", [])) >= 2:
        return True, "Model tried multiple approaches after error"
    return False, "Model did not handle permission error"

def _eval_recover_import_error(ws, answer, meta):
    p = ws / "imports.py"
    if not p.is_file():
        return False, "imports.py not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "import nonexistent_module" in content:
        return False, "Bad import not fixed"
    if "import" in content:
        return True, "Import error was fixed"
    return False, "File was modified but imports unclear"

def _eval_recover_git_conflict(ws, answer, meta):
    p = ws / "README.md"
    if not p.is_file():
        return False, "README.md not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    if "<<<<<<<" in content or ">>>>>>>" in content or "=======" in content:
        return False, "Git conflict markers still present"
    return True, "Git conflict was resolved"

def _eval_recover_type_error(ws, answer, meta):
    p = ws / "types.py"
    if not p.is_file():
        return False, "types.py not found"
    content = p.read_text(encoding="utf-8", errors="ignore")
    has_cast = "str(" in content or "int(" in content or "float(" in content
    if has_cast:
        return True, "Type error fixed with explicit cast"
    return False, "Type error not fixed"

def _eval_verify_tests_pass(ws, answer, meta):
    if _answer_contains(answer, "pass") and not _answer_contains(answer, "fail"):
        if _tool_was_used(meta, "shell") or _tool_was_used(meta, "file_read"):
            return True, "Model ran tests and confirmed they pass"
    if _answer_contains(answer, "all tests pass") or _answer_contains(answer, "tests passed"):
        return True, "Model confirmed tests pass"
    return False, "Model did not verify tests pass"

def _eval_verify_file_content(ws, answer, meta):
    if _answer_contains(answer, "correct") or _answer_contains(answer, "verified") or _answer_contains(answer, "matches"):
        if _tool_was_used(meta, "file_read"):
            return True, "Model read and verified file content"
    return False, "Model did not verify file content"

def _eval_verify_git_clean(ws, answer, meta):
    if _tool_was_used(meta, "git") or _tool_was_used(meta, "shell"):
        if _answer_contains(answer, "clean") or _answer_contains(answer, "nothing to commit") or _answer_contains(answer, "up to date"):
            return True, "Model verified git working tree is clean"
        return False, "Model checked git but did not confirm clean state"
    return False, "Model did not use git or shell to check status"

def _eval_verify_syntax(ws, answer, meta):
    if _answer_contains(answer, "valid") or _answer_contains(answer, "no syntax error") or _answer_contains(answer, "compiles"):
        return True, "Model verified syntax is valid"
    if _answer_contains(answer, "syntax error") and _answer_contains(answer, "found") or _answer_contains(answer, "line"):
        return True, "Model identified the syntax error"
    return False, "Model did not verify syntax"

def _eval_verify_checksum(ws, answer, meta):
    if _tool_was_used(meta, "shell"):
        if _answer_contains(answer, "match") or _answer_contains(answer, "verified") or _answer_contains(answer, "correct"):
            return True, "Model computed and verified checksum"
        if "sha256" in answer.lower() or "md5" in answer.lower() or "checksum" in answer.lower():
            return True, "Model computed checksum"
    return False, "Model did not verify checksum"

def _eval_verify_dependencies(ws, answer, meta):
    if _answer_contains(answer, "all") and _answer_contains(answer, "available"):
        return True, "Model verified all dependencies are available"
    if _answer_contains(answer, "missing") and _answer_contains(answer, "module"):
        return True, "Model identified missing dependencies"
    return False, "Model did not verify dependencies"

def _eval_verify_formatting(ws, answer, meta):
    if _tool_was_used(meta, "shell") or _tool_was_used(meta, "file_read"):
        if _answer_contains(answer, "formatted") or _answer_contains(answer, "pep 8") or _answer_contains(answer, "compliant"):
            return True, "Model verified formatting"
        return False, "Model checked but did not confirm formatting"
    return False, "Model did not check formatting"

def _eval_verify_no_secrets(ws, answer, meta):
    if _tool_was_used(meta, "grep") or _tool_was_used(meta, "shell"):
        if _answer_contains(answer, "no secret") or _answer_contains(answer, "clean") or _answer_contains(answer, "found"):
            return True, "Model searched for secrets and reported"
        return False, "Model searched but did not report findings"
    return False, "Model did not search for secrets"

def _eval_project_setup(ws, answer, meta):
    has_src = (ws / "src").is_dir()
    has_tests = (ws / "tests").is_dir()
    has_readme = _file_exists(ws, "README.md")
    has_init = _file_exists(ws, "src/__init__.py") or _file_exists(ws, "src/main.py")
    score = sum([has_src, has_tests, has_readme, has_init])
    if score >= 3:
        return True, f"Project structure created ({score}/4 components)"
    return False, f"Project structure incomplete ({score}/4 components)"

def _eval_refactor_module(ws, answer, meta):
    py_files = list(ws.glob("*.py"))
    if len(py_files) >= 2:
        return True, f"Module was refactored into {len(py_files)} files"
    return False, "Module was not split into multiple files"

def _eval_ci_pipeline(ws, answer, meta):
    ci_files = [
        ws / ".github" / "workflows" / "ci.yml",
        ws / ".gitlab-ci.yml",
        ws / ".circleci" / "config.yml",
    ]
    for f in ci_files:
        if f.is_file():
            content = f.read_text(encoding="utf-8", errors="ignore")
            if "test" in content.lower() or "lint" in content.lower() or "build" in content.lower():
                return True, f"CI pipeline created at {f.relative_to(ws)}"
    return False, "No CI configuration file was created"

def _eval_full_stack_feature(ws, answer, meta):
    has_backend = any(_file_matches(ws, str(p), r"(def |function |route|app\.)") for p in list(ws.glob("*.py")) + list(ws.glob("*.js")) + list(ws.glob("*.ts")))
    has_test = bool(list(ws.glob("test_*.py")) + list(ws.glob("*_test.*")))
    if has_backend and has_test:
        return True, "Full-stack feature with tests created"
    if has_backend:
        return False, "Backend created but no tests"
    return False, "Backend was not created"

def _eval_migration_script(ws, answer, meta):
    migration_files = list(ws.glob("migrate*.py")) + list(ws.glob("migration*.py")) + list(ws.glob("*.sql"))
    if migration_files:
        content = migration_files[0].read_text(encoding="utf-8", errors="ignore")
        if "create" in content.lower() or "alter" in content.lower() or "table" in content.lower():
            return True, f"Migration script created: {migration_files[0].name}"
    return False, "No migration script was created"

def _eval_documentation(ws, answer, meta):
    doc_files = list(ws.glob("*.md")) + list(ws.glob("docs/**/*.md"))
    if doc_files:
        content = doc_files[0].read_text(encoding="utf-8", errors="ignore")
        if len(content) > 100:
            return True, f"Documentation created: {doc_files[0].name}"
    return False, "No documentation was created"

def _eval_deploy_script(ws, answer, meta):
    deploy_files = (
        list(ws.glob("deploy*.sh")) + list(ws.glob("deploy*.py")) +
        list(ws.glob("Dockerfile")) + list(ws.glob("docker-compose*.yml"))
    )
    if deploy_files:
        return True, f"Deploy script created: {deploy_files[0].name}"
    return False, "No deployment script was created"

def _eval_api_with_docs(ws, answer, meta):
    has_api = any(_file_matches(ws, str(p), r"(@app\.|@router\.|def |FastAPI|Flask)") for p in ws.glob("*.py"))
    has_docs = bool(list(ws.glob("*.md")))
    if has_api and has_docs:
        return True, "API with documentation created"
    if has_api:
        return False, "API created but no documentation"
    return False, "API was not created"

def _eval_uses_file_read(ws, answer, meta):
    if _tool_was_used(meta, "file_read"):
        return True, "Model correctly used file_read"
    return False, f"Model did not use file_read. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_grep(ws, answer, meta):
    if _tool_was_used(meta, "grep"):
        return True, "Model correctly used grep"
    if _tool_was_used(meta, "shell") and _answer_contains(answer, "rg "):
        return True, "Model used shell with rg (equivalent to grep)"
    return False, f"Model did not use grep. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_find_file(ws, answer, meta):
    if _tool_was_used(meta, "find_file"):
        return True, "Model correctly used find_file"
    if _tool_was_used(meta, "shell") and ("find " in answer or "fd " in answer):
        return True, "Model used shell with find/fd (equivalent)"
    return False, f"Model did not use find_file. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_git(ws, answer, meta):
    if _tool_was_used(meta, "git"):
        return True, "Model correctly used git tool"
    if _tool_was_used(meta, "shell") and "git " in answer.lower():
        return True, "Model used shell with git (equivalent)"
    return False, f"Model did not use git. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_web_search(ws, answer, meta):
    if _tool_was_used(meta, "web_search"):
        return True, "Model correctly used web_search"
    return False, f"Model did not use web_search. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_file_write(ws, answer, meta):
    if _tool_was_used(meta, "file_write"):
        return True, "Model correctly used file_write"
    if _tool_was_used(meta, "shell") and _answer_contains(answer, ">") or _answer_contains(answer, "tee"):
        return True, "Model used shell to write file (equivalent)"
    return False, f"Model did not use file_write. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_file_edit(ws, answer, meta):
    if _tool_was_used(meta, "file_edit"):
        return True, "Model correctly used file_edit"
    if _tool_was_used(meta, "file_write") and _tool_was_used(meta, "file_read"):
        return True, "Model used read+write to edit (equivalent)"
    return False, f"Model did not use file_edit. Tools used: {meta.get('tools_used', [])}"

def _eval_uses_todo_write(ws, answer, meta):
    if _tool_was_used(meta, "todo_write"):
        return True, "Model correctly used todo_write for planning"
    return False, f"Model did not use todo_write. Tools used: {meta.get('tools_used', [])}"

def _eval_empty_input(ws, answer, meta):
    if _answer_contains(answer, "empty") or _answer_contains(answer, "no content") or _answer_contains(answer, "0 lines"):
        return True, "Model correctly identified empty file"
    if _tool_was_used(meta, "file_read"):
        return True, "Model read the empty file without crashing"
    return False, "Model did not handle empty file"

def _eval_unicode_handling(ws, answer, meta):
    if _answer_contains(answer, "héllo") or _answer_contains(answer, "wörld") or _answer_contains(answer, "日本"):
        return True, "Model correctly handled unicode content"
    if _tool_was_used(meta, "file_read"):
        return True, "Model read unicode file without error"
    return False, "Model did not handle unicode"

def _eval_large_file(ws, answer, meta):
    if _answer_contains(answer, "1000") or _answer_contains(answer, "large") or _answer_contains(answer, "truncat"):
        return True, "Model handled large file"
    if _tool_was_used(meta, "file_read") or _tool_was_used(meta, "shell"):
        return True, "Model processed large file without crashing"
    return False, "Model did not handle large file"

def _eval_special_chars_filename(ws, answer, meta):
    if _answer_contains(answer, "found") or _answer_contains(answer, "read") or _answer_contains(answer, "content"):
        return True, "Model handled special character filename"
    if _tool_was_used(meta, "file_read"):
        return True, "Model attempted to read file with special chars"
    return False, "Model did not handle special character filename"

def _eval_deeply_nested(ws, answer, meta):
    target = ws / "a" / "b" / "c" / "d" / "target.txt"
    if target.is_file():
        return True, "Model navigated deeply nested structure"
    if _tool_was_used(meta, "find_file") or _tool_was_used(meta, "shell"):
        return True, "Model searched for deeply nested file"
    return False, "Model did not find deeply nested file"

def _eval_binary_file(ws, answer, meta):
    if _answer_contains(answer, "binary") or _answer_contains(answer, "not text") or _answer_contains(answer, "cannot read"):
        return True, "Model correctly identified binary file"
    if _tool_was_used(meta, "file_read") or _tool_was_used(meta, "shell"):
        return True, "Model attempted to read binary file"
    return False, "Model did not handle binary file"

# Test case list: (id, category, prompt, expected_mode, expected_tools, eval_fn, setup_fn, max_steps, difficulty)
TEST_CASES = [
    # Category A: Simple file operations (FAST)
    ("fast_create_file", "simple_file_ops", "Create a file called hello.txt with the content 'Hello, World!' in the current workspace.", "FAST", ["file_write"], _eval_create_file, None, 15, 2),
    ("fast_read_file", "simple_file_ops", "Read the file target.txt and tell me what the secret value is.", "FAST", ["file_read"], _eval_read_and_report, _setup_read_file, 15, 2),
    ("fast_list_files", "simple_file_ops", "List all files in the current workspace and tell me what files are present.", "FAST", ["file_list"], _eval_list_files, _setup_list_files, 15, 2),
    ("fast_find_python", "simple_file_ops", "Find all Python files in the current workspace.", "FAST", ["find_file"], _eval_find_python_files, _setup_find_python, 15, 3),
    ("fast_write_config", "simple_file_ops", 'Create a JSON config file at config.json with the following content: {"name": "test_app", "version": "1.0.0", "debug": true}', "FAST", ["file_write"], _eval_write_config, None, 15, 2),
    ("fast_edit_file", "simple_file_ops", "Edit the file todo.py to replace 'TODO: implement' with 'Function implemented successfully'.", "FAST", ["file_edit"], _eval_edit_file, _setup_edit_file, 15, 3),
    ("fast_create_dir", "simple_file_ops", "Create a directory structure src/components in the current workspace.", "FAST", ["shell", "file_write"], _eval_create_directory, None, 15, 2),
    ("fast_copy_file", "simple_file_ops", "Copy the contents of original.txt to a new file called backup.txt.", "FAST", ["file_read", "file_write"], _eval_copy_file, _setup_copy_file, 15, 3),
    ("fast_count_lines", "simple_file_ops", "Count the number of lines in data.txt and report the total.", "FAST", ["shell", "file_read"], _eval_count_lines, _setup_count_lines, 15, 2),
    ("fast_append_file", "simple_file_ops", "Append a new entry 'new entry: task completed' to the file log.txt.", "FAST", ["file_write", "file_edit"], _eval_append_to_file, _setup_append_file, 15, 3),
    # Category B: Complex coding (THINK)
    ("think_binary_search", "complex_coding", "Write a Python implementation of binary search. The function should take a sorted list and a target value, returning the index or -1. Save it to binary_search.py.", "THINK", ["file_write"], _eval_binary_search, None, 20, 6),
    ("think_flatten", "complex_coding", "Write a Python function called flatten that takes a nested list of arbitrary depth and returns a flat list. Handle edge cases like empty lists and non-list elements. Save to flatten.py.", "THINK", ["file_write"], _eval_flatten_function, None, 20, 6),
    ("think_rest_api", "complex_coding", "Create a simple REST API using Python (Flask or FastAPI) with a /users endpoint that supports GET (list all) and POST (create new). Save to app.py.", "THINK", ["file_write"], _eval_rest_api, None, 20, 7),
    ("think_class_hierarchy", "complex_coding", "Create a Python class hierarchy in shapes.py: an abstract base class Shape with an abstract area() method, and concrete subclasses Circle (radius) and Rectangle (width, height) that implement area().", "THINK", ["file_write"], _eval_class_hierarchy, None, 20, 6),
    ("think_decorator", "complex_coding", "Write a Python retry decorator that retries a function up to 3 times on exception, with exponential backoff. Save to decorators.py.", "THINK", ["file_write"], _eval_decorator_pattern, None, 20, 7),
    ("think_data_pipeline", "complex_coding", "Write a data processing pipeline in pipeline.py that reads a JSON file, transforms each record (filter invalid, normalize fields), and writes the result to an output JSON file. Include error handling.", "THINK", ["file_write"], _eval_data_pipeline, None, 20, 7),
    ("think_test_suite", "complex_coding", "Write a comprehensive test suite for a function that validates email addresses. The function should check format, domain, and TLD. Create the validator and its tests.", "THINK", ["file_write"], _eval_test_suite, None, 20, 6),
    ("think_optimize", "complex_coding", "Write an optimized Python solution to find the first duplicate in a list. The solution should be O(n) time complexity. Save to optimized.py.", "THINK", ["file_write"], _eval_algorithm_optimize, None, 20, 7),
    # Category C: Error recovery (RECOVER)
    ("recover_missing_file", "error_recovery", "Read the file data.txt and display its contents. If the file doesn't exist, create it with some default content.", "RECOVER", ["file_read", "file_write"], _eval_recover_missing_file, None, 15, 5),
    ("recover_syntax_error", "error_recovery", "The file broken.py has a syntax error. Find and fix it so the file is valid Python.", "RECOVER", ["file_read", "file_edit", "shell"], _eval_recover_syntax_error, _setup_syntax_error, 15, 5),
    ("recover_wrong_path", "error_recovery", "Read the file at config.json in the workspace. If you can't find it at the root, search for it.", "RECOVER", ["file_read", "find_file", "grep"], _eval_recover_wrong_path, _setup_wrong_path, 15, 5),
    ("recover_test_failure", "error_recovery", "Run the tests in test_calc.py. If they fail, fix the code in calc.py so all tests pass.", "RECOVER", ["shell", "file_read", "file_edit"], _eval_recover_test_failure, _setup_test_failure, 20, 6),
    ("recover_permission", "error_recovery", "Try to write to /system/test.txt. If you get a permission denied error, write to test.txt in the current workspace instead.", "RECOVER", ["file_write", "shell"], _eval_recover_permission_denied, None, 15, 5),
    ("recover_import_error", "error_recovery", "The file imports.py has an import error. Run it to see the error, then fix the import so the file runs without error.", "RECOVER", ["shell", "file_read", "file_edit"], _eval_recover_import_error, _setup_import_error, 15, 5),
    ("recover_git_conflict", "error_recovery", "The file README.md has git merge conflict markers. Resolve the conflict by keeping both versions (combine them) and remove the conflict markers.", "RECOVER", ["file_read", "file_edit"], _eval_recover_git_conflict, _setup_git_conflict, 15, 6),
    ("recover_type_error", "error_recovery", "The file types.py has a type error. Run it to see the error, then fix it by adding explicit type conversion.", "RECOVER", ["shell", "file_read", "file_edit"], _eval_recover_type_error, _setup_type_error, 15, 5),
    # Category D: Verification (VERIFY)
    ("verify_tests_pass", "verification", "Run the test suite (test_math.py) and verify that all tests pass. Report the results.", "VERIFY", ["shell"], _eval_verify_tests_pass, _setup_verify_tests, 15, 4),
    ("verify_file_content", "verification", "Read config.yaml and verify it contains the correct database configuration (host: localhost, port: 5432).", "VERIFY", ["file_read"], _eval_verify_file_content, _setup_verify_file, 15, 4),
    ("verify_git_clean", "verification", "Check the git status of the current workspace and verify the working tree is clean (no uncommitted changes).", "VERIFY", ["git"], _eval_verify_git_clean, None, 15, 4),
    ("verify_syntax", "verification", "Check if check_me.py is valid Python (no syntax errors). Report whether it compiles successfully.", "VERIFY", ["shell", "file_read"], _eval_verify_syntax, _setup_verify_syntax, 15, 3),
    ("verify_checksum", "verification", "Compute the SHA256 checksum of important.bin and verify it matches the expected value. Report the checksum.", "VERIFY", ["shell"], _eval_verify_checksum, _setup_verify_checksum, 15, 4),
    ("verify_dependencies", "verification", "Check if all imports in app.py are available (can be imported). Report any missing modules.", "VERIFY", ["shell", "file_read"], _eval_verify_dependencies, _setup_verify_deps, 15, 4),
    ("verify_formatting", "verification", "Check if style.py follows PEP 8 formatting conventions. Report any formatting issues found.", "VERIFY", ["shell", "file_read"], _eval_verify_formatting, _setup_verify_formatting, 15, 4),
    ("verify_no_secrets", "verification", "Search settings.py for any hardcoded secrets, API keys, or passwords. Report what you find.", "VERIFY", ["grep", "file_read"], _eval_verify_no_secrets, _setup_verify_secrets, 15, 4),
    # Category E: Multi-step planning (THINK)
    ("plan_project_setup", "multi_step_planning", "Set up a new Python project with the following structure: src/ directory with __init__.py and main.py, tests/ directory with __init__.py, and a README.md with project description.", "THINK", ["file_write", "shell", "todo_write"], _eval_project_setup, None, 20, 6),
    ("plan_refactor", "multi_step_planning", "Refactor monolith.py by splitting the classes (User, Product, Order) into separate files: user.py, product.py, order.py. Update any imports accordingly.", "THINK", ["file_read", "file_write", "file_edit"], _eval_refactor_module, _setup_refactor, 20, 7),
    ("plan_ci_pipeline", "multi_step_planning", "Create a CI/CD pipeline configuration that runs tests and linting on every push. Use GitHub Actions format. Save to .github/workflows/ci.yml.", "THINK", ["file_write"], _eval_ci_pipeline, None, 20, 6),
    ("plan_full_stack", "multi_step_planning", "Create a simple todo API with: a backend endpoint (GET /todos, POST /todos), a data model, and unit tests. Save all files in the workspace.", "THINK", ["file_write", "todo_write"], _eval_full_stack_feature, None, 25, 7),
    ("plan_migration", "multi_step_planning", "Write a database migration script that creates a users table with columns: id (primary key), email (unique), name, created_at. Save as migrate.py or migration.sql.", "THINK", ["file_write"], _eval_migration_script, None, 20, 6),
    ("plan_documentation", "multi_step_planning", "Create comprehensive documentation for a Python project: a README.md with installation instructions, usage examples, and API documentation.", "THINK", ["file_write"], _eval_documentation, None, 20, 5),
    ("plan_deploy", "multi_step_planning", "Create a deployment script that: builds the project, runs tests, and if tests pass, creates a deployable package. Save as deploy.sh or Dockerfile.", "THINK", ["file_write"], _eval_deploy_script, None, 20, 6),
    ("plan_api_with_docs", "multi_step_planning", "Create a simple REST API with at least one endpoint, and write API documentation describing the endpoint, parameters, and response format.", "THINK", ["file_write"], _eval_api_with_docs, None, 25, 7),
    # Category F: Tool selection
    ("tool_select_file_read", "tool_selection", "Read the contents of target.txt and tell me what it says.", "FAST", ["file_read"], _eval_uses_file_read, _setup_read_file, 15, 2),
    ("tool_select_grep", "tool_selection", "Search for all occurrences of the word 'import' in Python files in the current workspace.", "FAST", ["grep"], _eval_uses_grep, _setup_verify_deps, 15, 3),
    ("tool_select_find_file", "tool_selection", "Find all files matching the pattern '*.py' in the current workspace.", "FAST", ["find_file"], _eval_uses_find_file, _setup_find_python, 15, 3),
    ("tool_select_git", "tool_selection", "Show the git log for the current repository, last 5 commits.", "FAST", ["git"], _eval_uses_git, None, 15, 3),
    ("tool_select_web_search", "tool_selection", "Search the web for the latest Python release version.", "FAST", ["web_search"], _eval_uses_web_search, None, 15, 3),
    ("tool_select_file_write", "tool_selection", "Write the text 'Hello from the eval suite' to a file called output.txt.", "FAST", ["file_write"], _eval_uses_file_write, None, 15, 2),
    ("tool_select_file_edit", "tool_selection", "Edit the file todo.py to change 'TODO: implement' to 'DONE: implemented'.", "FAST", ["file_edit"], _eval_uses_file_edit, _setup_edit_file, 15, 3),
    ("tool_select_todo_write", "tool_selection", "Create a todo list with 5 items for building a web application: design, backend, frontend, testing, deployment. Track them with the todo tool.", "THINK", ["todo_write"], _eval_uses_todo_write, None, 15, 4),
    # Category G: Edge cases
    ("edge_empty_file", "edge_cases", "Read the file empty.txt and describe what you find. Handle the case appropriately.", "VERIFY", ["file_read"], _eval_empty_input, _setup_empty_file, 15, 3),
    ("edge_unicode", "edge_cases", "Read the file unicode.txt which contains unicode text and report its contents.", "FAST", ["file_read"], _eval_unicode_handling, _setup_unicode_file, 15, 3),
    ("edge_large_file", "edge_cases", "Read the file large.txt (it has 1000 lines) and report how many lines it has.", "FAST", ["shell", "file_read"], _eval_large_file, _setup_large_file, 15, 4),
    ("edge_special_chars", "edge_cases", "Read the file 'file with spaces & symbols!.txt' and report its contents.", "FAST", ["file_read"], _eval_special_chars_filename, _setup_special_chars, 15, 4),
    ("edge_nested_dirs", "edge_cases", "Find the file target.txt that is somewhere in a deeply nested directory structure. Read it and report its contents.", "THINK", ["find_file", "file_read"], _eval_deeply_nested, _setup_nested_dirs, 15, 5),
    ("edge_binary_file", "edge_cases", "Read the file data.bin and determine if it is a text file or a binary file. Report your findings.", "VERIFY", ["file_read", "shell"], _eval_binary_file, _setup_binary_file, 15, 4),
]

# Filter by categories if specified
if TEST_CATEGORIES:
    TEST_CASES = [tc for tc in TEST_CASES if tc[1] in TEST_CATEGORIES]

print(f"Loaded {len(TEST_CASES)} test cases across {len(set(tc[1] for tc in TEST_CASES))} categories")
for cat in sorted(set(tc[1] for tc in TEST_CASES)):
    count = sum(1 for tc in TEST_CASES if tc[1] == cat)
    print(f"  {cat:<30s} {count:3d}")

## Cell 5: Run Evaluation — v4.1

For each test case:
- Send the prompt to the model (via Together API or local T4)
- Parse the response for control token, tool calls, task completion
- Record pass/fail per category
- Run `NUM_SEEDS` times and average

In [ ]:
# --------------------------------------------------------------------------- #
# Tool runner (simplified, sandboxed)                                          #
# --------------------------------------------------------------------------- #

class ToolRunner:
    """Executes tool calls within a sandboxed workspace directory."""

    def __init__(self, workspace):
        self.workspace = workspace

    def execute(self, name, args):
        try:
            if name == "shell":
                return self._shell(args)
            elif name == "file_read":
                return self._file_read(args)
            elif name == "file_write":
                return self._file_write(args)
            elif name == "file_edit":
                return self._file_edit(args)
            elif name == "file_list":
                return self._file_list(args)
            elif name == "grep":
                return self._grep(args)
            elif name == "find_file":
                return self._find_file(args)
            elif name == "git":
                return self._git(args)
            elif name == "web_search":
                return (f"(web search for '{args.get('query', '')}' -- simulated in eval mode)", True, None)
            elif name == "web_fetch":
                return self._web_fetch(args)
            elif name == "todo_write":
                return self._todo_write(args)
            elif name == "ask_user":
                return ("(auto-approved: proceed with your best judgment)", True, None)
            else:
                return (f"unknown tool: {name}", False, f"unknown tool: {name}")
        except Exception as e:
            return (f"tool error: {e}", False, str(e))

    def _resolve(self, path):
        p = Path(path)
        if not p.is_absolute():
            p = self.workspace / p
        try:
            p.resolve().relative_to(self.workspace.resolve())
        except ValueError:
            return self.workspace / Path(path).name
        return p

    def _shell(self, args):
        cmd = args.get("command", "")
        if not cmd:
            return ("", False, "missing 'command'")
        timeout = min(args.get("timeout", 30), 30)
        try:
            proc = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=str(self.workspace))
            output = proc.stdout
            if proc.stderr:
                output += f"\n[stderr]\n{proc.stderr}"
            if len(output) > 8000:
                output = output[:8000] + f"\n... (truncated, {len(output)} total chars)"
            return (output, proc.returncode == 0, None if proc.returncode == 0 else f"exit code {proc.returncode}")
        except subprocess.TimeoutExpired:
            return (f"command timed out after {timeout}s", False, "timeout")

    def _file_read(self, args):
        path = args.get("path", "")
        if not path:
            return ("", False, "missing 'path'")
        p = self._resolve(path)
        if not p.is_file():
            return (f"not a file: {path}", False, f"not a file: {path}")
        content = p.read_text(encoding="utf-8", errors="ignore")
        if len(content) > 50000:
            content = content[:50000] + f"\n... (truncated, {len(content)} total chars)"
        return (content, True, None)

    def _file_write(self, args):
        path = args.get("path", "")
        content = args.get("content", "")
        if not path:
            return ("", False, "missing 'path'")
        p = self._resolve(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        return (f"wrote {len(content)} chars to {path}", True, None)

    def _file_edit(self, args):
        path = args.get("path", "")
        old = args.get("old_string", "")
        new = args.get("new_string", "")
        if not path or not old:
            return ("", False, "missing 'path' or 'old_string'")
        p = self._resolve(path)
        if not p.is_file():
            return (f"not a file: {path}", False, f"not a file: {path}")
        content = p.read_text(encoding="utf-8", errors="ignore")
        count = content.count(old)
        if count == 0:
            return ("old_string not found", False, "old_string not found")
        if count > 1:
            return (f"old_string found {count} times", False, f"old_string found {count} times, need unique match")
        content = content.replace(old, new, 1)
        p.write_text(content, encoding="utf-8")
        return (f"edited {path}", True, None)

    def _file_list(self, args):
        path = args.get("path", ".")
        p = self._resolve(path)
        if not p.is_dir():
            return (f"not a directory: {path}", False, f"not a directory: {path}")
        entries = sorted(str(x.name) + ("/" if x.is_dir() else "") for x in p.iterdir())
        return ("\n".join(entries[:500]), True, None)

    def _grep(self, args):
        pattern = args.get("pattern", "")
        if not pattern:
            return ("", False, "missing 'pattern'")
        path = args.get("path", ".")
        p = self._resolve(path)
        try:
            proc = subprocess.run(f"grep -rn {pattern} {p}", shell=True, capture_output=True, text=True, timeout=30)
            return (proc.stdout or "(no matches)", True, None)
        except Exception as e:
            return (f"search error: {e}", False, str(e))

    def _find_file(self, args):
        pattern = args.get("pattern", "")
        if not pattern:
            return ("", False, "missing 'pattern'")
        path = args.get("path", ".")
        p = self._resolve(path)
        results = sorted(str(x) for x in p.glob(pattern) if x.is_file())
        return ("\n".join(results[:500]) if results else "(no files found)", True, None)

    def _git(self, args):
        subcmd = args.get("command", "")
        if not subcmd:
            return ("", False, "missing 'command'")
        repo = args.get("repo", ".")
        p = self._resolve(repo)
        try:
            proc = subprocess.run(f"git {subcmd}", shell=True, cwd=str(p), capture_output=True, text=True, timeout=30)
            output = proc.stdout
            if proc.stderr:
                output += f"\n[stderr]\n{proc.stderr}"
            return (output, proc.returncode == 0, None if proc.returncode == 0 else f"exit code {proc.returncode}")
        except subprocess.TimeoutExpired:
            return ("git command timed out", False, "timeout")

    def _web_fetch(self, args):
        url = args.get("url", "")
        if not url:
            return ("", False, "missing 'url'")
        try:
            import urllib.request
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=15) as resp:
                content = resp.read().decode("utf-8", errors="ignore")
            return (content[:8000], True, None)
        except Exception as e:
            return (f"fetch error: {e}", False, str(e))

    def _todo_write(self, args):
        todos = args.get("todos", [])
        if not isinstance(todos, list):
            return ("", False, "invalid 'todos'")
        lines = []
        for i, t in enumerate(todos, 1):
            content = t.get("content", "") if isinstance(t, dict) else str(t)
            status = t.get("status", "pending") if isinstance(t, dict) else "pending"
            icon = {"completed": "[x]", "in_progress": "[~]", "pending": "[ ]"}.get(status, "[ ]")
            lines.append(f"{i}. {icon} {content}")
        return ("\n".join(lines) if lines else "(empty todo list)", True, None)


# --------------------------------------------------------------------------- #
# Agent loop                                                                   #
# --------------------------------------------------------------------------- #

class AgentLoop:
    """Simplified agent loop for evaluation."""

    def __init__(self, client, model, workspace, max_steps=15, timeout_s=120, temperature=0.0, max_tokens=2048):
        self.client = client
        self.model = model
        self.workspace = workspace
        self.max_steps = max_steps
        self.timeout_s = timeout_s
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.runner = ToolRunner(workspace)

    def run(self, prompt, tools_schema):
        system_prompt = build_system_prompt(tools_schema)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
        modes_used = []
        tools_used = []
        tool_calls_made = []
        step_records = []
        answer = ""
        error = None
        step = 0
        start_time = time.monotonic()

        while step < self.max_steps:
            if time.monotonic() - start_time > self.timeout_s:
                error = f"timeout after {time.monotonic() - start_time:.1f}s"
                break
            step += 1
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                )
            except Exception as e:
                error = f"model error at step {step}: {e}"
                break

            choice = resp.choices[0]
            raw_text = choice.message.content or ""

            mode, text = parse_control_token(raw_text)
            if mode:
                modes_used.append(mode)

            tool_calls = parse_tool_calls(text)

            step_record = {"step": step, "mode": mode, "text": text[:500], "tool_calls": [], "tool_results": [], "error": None}

            if tool_calls:
                messages.append({"role": "assistant", "content": raw_text})
                for tc in tool_calls:
                    tc_record = {"name": tc["name"], "arguments": tc["arguments"], "valid": tc["valid"]}
                    tool_calls_made.append(tc_record)
                    step_record["tool_calls"].append(tc_record)
                    if tc["valid"]:
                        tools_used.append(tc["name"])
                        result_text, result_ok, result_error = self.runner.execute(tc["name"], tc["arguments"])
                        step_record["tool_results"].append({"ok": result_ok, "output": result_text[:500], "error": result_error})
                        messages.append({"role": "tool", "content": result_text, "tool_call_id": tc["id"], "name": tc["name"]})
                    else:
                        step_record["tool_results"].append({"ok": False, "output": "", "error": f"invalid tool call format: {tc.get('parse_error', 'unknown')}"})
                        messages.append({"role": "tool", "content": "Error: invalid tool call format. Please use the correct format.", "tool_call_id": tc["id"], "name": tc["name"]})
            else:
                answer = text
                messages.append({"role": "assistant", "content": raw_text})
                step_records.append(step_record)
                break

            step_records.append(step_record)

        return {
            "answer": answer,
            "modes_used": modes_used,
            "tools_used": tools_used,
            "tool_calls_made": tool_calls_made,
            "steps": step,
            "step_records": step_records,
            "error": error,
        }


# --------------------------------------------------------------------------- #
# Inference backends                                                           #
# --------------------------------------------------------------------------- #

def get_together_client():
    """Get OpenAI client configured for Together AI."""
    from openai import OpenAI
    return OpenAI(
        base_url=TOGETHER_BASE_URL,
        api_key=TOGETHER_API_KEY,
        timeout=120.0,
        max_retries=3,
    )

_local_model = None
_local_tokenizer = None

def load_local_model():
    """Load the model locally in 4-bit NF4 for T4 inference."""
    global _local_model, _local_tokenizer
    if _local_model is not None:
        return _local_model, _local_tokenizer
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print(f"Loading model from {MODEL_PATH} in 4-bit NF4...")
    _local_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    _local_model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print(f"Model loaded. Device: {_local_model.device}")
    return _local_model, _local_tokenizer


class LocalModelClient:
    """Wrapper to make local HF model look like an OpenAI client."""

    def __init__(self):
        self.model, self.tokenizer = load_local_model()

    class _Completions:
        def __init__(self, outer):
            self.outer = outer

        def create(self, model, messages, temperature=0.0, max_tokens=2048):
            import torch
            # Build prompt from messages
            prompt_parts = []
            for msg in messages:
                role = msg["role"]
                content = msg["content"]
                if role == "system":
                    prompt_parts.append(f"<|im_start|>system\n{content}<|im_end|>")
                elif role == "user":
                    prompt_parts.append(f"<|im_start|>user\n{content}<|im_end|>")
                elif role == "assistant":
                    prompt_parts.append(f"<|im_start|>assistant\n{content}<|im_end|>")
                elif role == "tool":
                    prompt_parts.append(f"<|im_start|>tool\n{content}<|im_end|>")
            prompt_parts.append("<|im_start|>assistant\n")
            full_prompt = "\n".join(prompt_parts)

            inputs = self.outer.tokenizer(full_prompt, return_tensors="pt").to(self.outer.model.device)
            with torch.no_grad():
                outputs = self.outer.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=max(temperature, 0.01),
                    do_sample=temperature > 0,
                    pad_token_id=self.outer.tokenizer.eos_token_id,
                )
            generated = outputs[0][inputs["input_ids"].shape[1]:]
            text = self.outer.tokenizer.decode(generated, skip_special_tokens=True)

            class _Choice:
                class _Message:
                    content = text
                message = _Message()
            class _Resp:
                choices = [_Choice()]
            return _Resp()

    @property
    def chat(self):
        return type("obj", (object,), {"completions": self._Completions(self)})()


def get_client():
    """Get the appropriate client based on EVAL_MODE."""
    if EVAL_MODE == "together_api":
        return get_together_client()
    elif EVAL_MODE == "local_t4":
        return LocalModelClient()
    else:
        raise ValueError(f"Unknown EVAL_MODE: {EVAL_MODE}")


# --------------------------------------------------------------------------- #
# Run evaluation for a single model                                            #
# --------------------------------------------------------------------------- #

def run_eval(model_name, client, label=""):
    """Run evaluation for a single model. Returns list of result dicts."""
    tools_schema = get_tool_schemas()
    all_results = []

    print(f"\n{'='*60}")
    print(f"  Running evaluation: {label or model_name}")
    print(f"  Mode: {EVAL_MODE}")
    print(f"  Cases: {len(TEST_CASES)} x {NUM_SEEDS} seeds = {len(TEST_CASES) * NUM_SEEDS} runs")
    print(f"{'='*60}\n")

    for i, tc in enumerate(TEST_CASES):
        case_id, category, prompt, expected_mode, expected_tools, eval_fn, setup_fn, max_steps, difficulty = tc
        seed_results = []
        for seed in range(NUM_SEEDS):
            # Create temp workspace
            workspace = Path(tempfile.mkdtemp(prefix=f"eval_{case_id}_s{seed}_"))

            # Run setup
            if setup_fn:
                try:
                    setup_fn(workspace)
                except Exception as e:
                    shutil.rmtree(workspace, ignore_errors=True)
                    seed_results.append({
                        "case_id": case_id, "category": category, "seed": seed,
                        "passed": False, "detail": f"setup error: {e}", "error": str(e),
                        "mode_correct": False, "tools_correct": False, "task_completed": False,
                        "expected_mode": expected_mode, "expected_tools": expected_tools,
                        "modes_used": [], "tools_used": [], "answer": "", "steps": 0,
                        "duration_s": 0.0, "score": 0.0,
                    })
                    continue

            # Run agent loop
            loop = AgentLoop(client, model_name, workspace, max_steps=max_steps, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
            start = time.monotonic()
            try:
                loop_result = loop.run(prompt, tools_schema)
            except Exception as e:
                loop_result = {"answer": "", "modes_used": [], "tools_used": [], "tool_calls_made": [], "steps": 0, "step_records": [], "error": str(e)}
            duration = time.monotonic() - start

            # Build metadata
            metadata = {
                "tools_used": loop_result["tools_used"],
                "modes_used": loop_result["modes_used"],
                "tool_calls": loop_result["tool_calls_made"],
                "raw_response": loop_result["answer"],
                "success": loop_result["error"] is None,
                "steps": loop_result["steps"],
                "workspace": workspace,
            }

            # Run eval function
            passed = False
            detail = "no eval function"
            if eval_fn:
                try:
                    passed, detail = eval_fn(workspace, loop_result["answer"], metadata)
                except Exception as e:
                    detail = f"eval function error: {e}"

            # Determine metrics
            first_mode = loop_result["modes_used"][0] if loop_result["modes_used"] else None
            mode_correct = first_mode is not None and first_mode.upper() == expected_mode.upper()
            expected_set = set(expected_tools)
            actual_set = set(loop_result["tools_used"])
            tools_correct = bool(expected_set & actual_set) if expected_set else True
            task_completed = loop_result["error"] is None and bool(loop_result["answer"])

            # Composite score
            score = 0.0
            if passed: score += 0.5
            if mode_correct: score += 0.2
            if tools_correct: score += 0.2
            if task_completed: score += 0.1

            seed_results.append({
                "case_id": case_id, "category": category, "seed": seed,
                "passed": passed, "detail": detail,
                "mode_correct": mode_correct, "tools_correct": tools_correct,
                "task_completed": task_completed,
                "expected_mode": expected_mode, "expected_tools": expected_tools,
                "modes_used": loop_result["modes_used"],
                "tools_used": loop_result["tools_used"],
                "answer": loop_result["answer"][:2000],
                "steps": loop_result["steps"],
                "duration_s": duration,
                "score": score,
                "error": loop_result["error"],
            })

            # Cleanup
            shutil.rmtree(workspace, ignore_errors=True)

        # Average across seeds
        avg_passed = sum(1 for r in seed_results if r["passed"]) / len(seed_results) if seed_results else 0
        avg_score = sum(r["score"] for r in seed_results) / len(seed_results) if seed_results else 0
        status = "PASS" if avg_passed > 0.5 else "FAIL"
        print(f"  [{i+1}/{len(TEST_CASES)}] {case_id:<30s} {status} ({avg_passed:.0%}) {sum(r['duration_s'] for r in seed_results):.1f}s")

        # Use the first seed result as representative, but update with averages
        rep = dict(seed_results[0])
        rep["seed_results"] = seed_results
        rep["avg_passed"] = avg_passed
        rep["avg_score"] = avg_score
        rep["passed"] = avg_passed > 0.5  # majority vote
        all_results.append(rep)

    return all_results


# Run v4.1 evaluation
print("\n" + "="*60)
print("  EVALUATING v4.1 MODEL")
print("="*60)

v41_client = get_client()
v41_model_name = V41_MODEL if EVAL_MODE == "together_api" else "local_t4_model"
v41_results = run_eval(v41_model_name, v41_client, label="v4.1")

print(f"\nv4.1 complete: {sum(1 for r in v41_results if r['passed'])}/{len(v41_results)} passed")

## Cell 6: Run Evaluation — v3.1 (baseline)

Same test cases, same setup, but using the v3.1 model as baseline.

In [ ]:
# Run v3.1 evaluation
print("\n" + "="*60)
print("  EVALUATING v3.1 BASELINE MODEL")
print("="*60)

# For together_api mode, v3.1 uses the same API endpoint
# For local_t4 mode, we can't load two 9B models simultaneously, so we use the API if available
if EVAL_MODE == "local_t4":
    if TOGETHER_API_KEY:
        print("  NOTE: Using Together API for v3.1 baseline (can't load two models on T4)")
        v31_client = get_together_client()
        v31_model_name = V31_MODEL
    else:
        print("  ERROR: Cannot run v3.1 baseline in local_t4 mode without TOGETHER_API_KEY")
        print("  Set TOGETHER_API_KEY to use the API for the v3.1 baseline comparison.")
        v31_results = []
        v31_model_name = V31_MODEL
else:
    v31_client = get_together_client()
    v31_model_name = V31_MODEL
    v31_results = run_eval(v31_model_name, v31_client, label="v3.1")

if v31_results:
    print(f"\nv3.1 complete: {sum(1 for r in v31_results if r['passed'])}/{len(v31_results)} passed")
else:
    print("\nv3.1 evaluation skipped (no API key for baseline).")

## Cell 7: Statistical Comparison

Compute:
- Wilson 95% confidence intervals for both models
- Fisher's exact test for significance
- Cohen's d effect size
- Per-category breakdown with Bonferroni correction
- Overall verdict: improvement, regression, or no change

In [ ]:
# Statistical comparison
print("\n" + "="*60)
print("  STATISTICAL COMPARISON")
print("="*60)

if v41_results and v31_results:
    stat_report = statistical_report(v41_results, v31_results)

    print(f"\n  Verdict: {stat_report['verdict'].upper()}")
    print(f"  v4.1 pass rate: {stat_report['overall']['v4']['pass_rate']:.1%}")
    print(f"  v3.1 pass rate: {stat_report['overall']['v3']['pass_rate']:.1%}")
    print(f"  Fisher p-value: {stat_report['overall']['fisher_exact']['p_value']:.4f}")
    print(f"  Cohen's d:      {stat_report['overall']['cohens_d']:.3f}")
    print()

    # Print full report
    print(format_report_text(stat_report))
else:
    stat_report = {"verdict": "incomplete", "error": "Missing results for one or both models"}
    print("  Cannot compute statistics: missing results for one or both models.")

## Cell 8: Save Results

Save the full results to Google Drive as JSON.

In [ ]:
# Save results to Drive
import datetime

output = {
    "v41_results": v41_results,
    "v31_results": v31_results,
    "statistical_report": stat_report,
    "config": {
        "eval_mode": EVAL_MODE,
        "together_api_key_set": bool(TOGETHER_API_KEY),
        "v41_model": V41_MODEL if EVAL_MODE == "together_api" else MODEL_PATH,
        "v31_model": V31_MODEL,
        "max_tokens": MAX_TOKENS,
        "temperature": TEMPERATURE,
        "num_seeds": NUM_SEEDS,
        "test_categories": TEST_CATEGORIES,
        "num_test_cases": len(TEST_CASES),
        "inference_note": (
            "fp16 on H100 via Together AI (fair comparison with v3.1)"
            if EVAL_MODE == "together_api"
            else "4-bit NF4 on T4 GPU (results may differ from fp16 H100 benchmarks)"
        ),
    },
    "timestamp": datetime.datetime.now().isoformat(),
}

os.makedirs(PROJECT_DIR, exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results saved to: {RESULTS_PATH}")
print(f"  v4.1 results: {len(v41_results)} cases")
print(f"  v3.1 results: {len(v31_results)} cases")
print(f"  Verdict: {stat_report.get('verdict', 'N/A')}")

## Cell 9: Display Summary Table

Print a formatted comparison table showing per-category and overall results.

In [ ]:
# Display summary table
if v41_results and v31_results:
    print(format_summary_table(v41_results, v31_results, stat_report))
else:
    print("Cannot display summary: missing results for one or both models.")
    if v41_results:
        v41_passed = sum(1 for r in v41_results if r['passed'])
        print(f"\nv4.1 only: {v41_passed}/{len(v41_results)} passed ({v41_passed/len(v41_results):.1%})")